In [6]:
f = 'asd__SA.md'
f.replace(".md", "").replace("_", " ")

'asd  SA'

In [2]:
from flask import Flask, render_template
from markupsafe import Markup
import markdown
import os

from charts.pnl_chart import generate_pnl_chart

from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler
from flask import Response

app = Flask(__name__)

PAGES_DIR = "pages"
clients = []

def render_md(filename):
    path = os.path.join(PAGES_DIR, filename)
    if not os.path.exists(path):
        return "<h1>Page not found</h1>"
    with open(path, "r", encoding="utf-8") as f:
        md = f.read()
    return markdown.markdown(md, extensions=["fenced_code", "tables"])

@app.route("/")
def index():
    pages = [f for f in os.listdir(PAGES_DIR) if f.endswith(".md")]
    return render_template("base.html", pages=pages, content="<h2>Select a page</h2>")

@app.route("/page/<filename>")
def page(filename):
    pages = [f for f in os.listdir(PAGES_DIR) if f.endswith(".md")]
    html = render_md(filename)
    return render_template("base.html", pages=pages, content=Markup(html))

@app.route("/chart")
def chart():
    pages = [f for f in os.listdir(PAGES_DIR) if f.endswith(".md")]
    chart_html = generate_pnl_chart()
    return render_template("base.html", pages=pages, content=Markup(chart_html))

# --- Live Reload (SSE) ---
class ChangeHandler(FileSystemEventHandler):
    def on_modified(self, event):
        for q in clients:
            q.put("reload")

@app.route("/events")
def events():
    def stream():
        from queue import Queue
        q = Queue()
        clients.append(q)
        try:
            while True:
                msg = q.get()
                yield f"data: {msg}\n\n"
        finally:
            clients.remove(q)
    return Response(stream(), mimetype="text/event-stream")

observer = Observer()
observer.schedule(ChangeHandler(), path=PAGES_DIR, recursive=True)
observer.start()

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=True)


In [4]:
print(generate_pnl_chart())

<div style="height:400px; width:100%;">                        <script>window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
        <script charset="utf-8" src="https://cdn.plot.ly/plotly-4.0.0.min.js" integrity="sha256-FEYfO0yRyLtZCpnW0Dw/0DHKQO7Afrq3ml4+rBB818o=" crossorigin="anonymous"></script>                <div id="3e73a65c-bc31-4560-b924-0de71e0856dc" class="plotly-graph-div" style="height:100%; width:100%;"></div>            <script>                window.PLOTLYENV=window.PLOTLYENV || {};                                if (document.getElementById("3e73a65c-bc31-4560-b924-0de71e0856dc")) {                    Plotly.newPlot(                        "3e73a65c-bc31-4560-b924-0de71e0856dc",                        [{"marker":{"color":["#4CAF50","#2196F3","#F44336"]},"x":["Paper Trading #1","Paper Trading #2","Tradestation - Equity"],"y":{"dtype":"f8","bdata":"KVyPwvUAdUAAAAAAALCEQMP1KFyPYmTA"},"type":"bar"}],                        {"template":{"data":{"barpolar":[{"marker":{"lin